In [ ]:
import re
import numpy as np
import pandas as pd

BASE = "/Users/parin/Desktop/prosecutor explorer/plymouth"
plymouth_charges = pd.read_parquet(f"{BASE}/plymouth-charges-enriched.parquet")
plymouth_extras = pd.read_parquet(f"{BASE}/plymouth-extras.parquet")

KEY = ["case_id", "charge_count_number"]
ch = plymouth_charges.drop_duplicates().copy()
ex = plymouth_extras.drop_duplicates(subset=KEY).copy()
ch["charge_count_number"] = ch.charge_count_number.astype("Int64")
ex["charge_count_number"] = ex.charge_count_number.astype("Int64")

df = ch.merge(ex[KEY + ["data_quality_tier", "universal_outcome",
                        "charge_type_description", "sentence_type_description"]],
              on=KEY, how="left", validate="m:1")
print(len(ch), "->", len(df))

In [ ]:
YEARS = (2017, 2024)
MISSING_MAX = 0.20
MIN_CELL = 30
POP = {y: 530_819 for y in range(2017, 2025)}   # 2020 Census; swap in annual PEP/ACS
EVENTS = {"MA CJ reform act": "2018-04-13", "COVID court closures": "2020-03-16"}
DEF_COL = "def_filled"

def pct(x): return (100 * x).round(1)
def share(s):
    t = s.value_counts(dropna=False)
    return pd.DataFrame({"n": t, "pct": pct(t / t.sum())})
def rate_by(frame, flag, by):
    t = frame.assign(f=np.asarray(flag)).groupby(by, observed=True).f.agg(n="size", k="sum")
    t["pct"] = pct(t.k / t.n)
    return t

In [ ]:
bad = (df.disposition_date < "1986-01-01") | (df.disposition_date > pd.Timestamp.today())
df.loc[bad, "disposition_date"] = pd.NaT

# case key: fill null defendant_id from the case's single known id; split multi-defendant dockets
g = df.groupby("case_id").defendant_id
n_ids = g.nunique()
df["def_filled"] = df.defendant_id.fillna(df.case_id.map(g.first().where(n_ids.eq(1))))
df["case_key"] = df.case_id + "|" + df.def_filled.fillna("UNK")

DISPO_MAP = {
    "DM": "dismissal", "NP": "dismissal", "NO": "dismissal", "NB": "dismissal", "DG": "dismissal", "D$": "dismissal",
    "DY": "declined", "DP": "declined", "NY": "declined",                 # pre-arraignment = declination proxy
    "DD": "diversion", "DC": "diversion", "PB": "diversion",
    "GP": "plea", "PF": "plea", "GL": "plea", "PL": "plea", "NC": "plea",
    "CW": "cwof",
    "GF": "trial_conviction", "GI": "trial_conviction", "VF": "trial_conviction", "FL": "trial_conviction",
    "NG": "acquittal", "DV": "acquittal", "NGI": "acquittal",
    "NR": "civil", "RF": "civil", "R$": "civil",
    "TF": "other", "FI": "other", "BO": "other", "RM": "other", "AW": "other",
    "AF": "other", "IN": "other", "NI": "other", "CC": "other",
}
LESSER_CODES = {"GL", "PL", "FL"}

df["outcome"] = df.disposition_code.map(DISPO_MAP)
df.loc[df.disposition_code.isna(), "outcome"] = "pending"
print("unmapped codes:", df.loc[df.outcome.isna(), "disposition_code"].value_counts().to_dict())
df["outcome"] = df.outcome.fillna("other")
df["lesser"] = df.disposition_code.isin(LESSER_CODES)

# severity: charge_category, backfilled from the enrichment degree fields
DEG_MAP = {"F": "F", "Felony": "F", "M+": "M+", "MP": "M+", "M-": "M-", "MM": "M-",
           "Misd.": "M", "CM": "CM", "CI": "CM", "Civil": "CM"}
df["sev_cat"] = df.charge_category
for col in ("mc_degree", "degree", "sc_degree"):
    df["sev_cat"] = df.sev_cat.fillna(df[col].map(DEG_MAP))

df["placeholder"] = df.charge_code.isin(["666666", "999999"])
df["is_civil"] = (df.sev_cat.eq("CM").fillna(False)
                  | df.charge_type_description.str.contains("CIVIL", case=False, na=False)
                  | df.charge_description.str.contains(r"\s\*(?:\s|$)", na=False)
                  | df.outcome.eq("civil"))
df["sev_rank"] = np.where(df.is_civil, 0, df.sev_cat.map({"F": 3, "M+": 2, "M": 1.5, "M-": 1}).fillna(0.5))
df["sev_class"] = np.select(
    [df.is_civil, df.sev_cat.eq("F").fillna(False), df.sev_cat.isin(["M+", "M", "M-"])],
    ["Civil", "Felony", "Misdemeanor"], "Unknown")

df["offense_cat"] = np.where(df.placeholder, "Other/Unknown", df.offense_group)
df["race"] = df.defendant_race.fillna("Unknown").replace({"American Indian/Native": "Other"})
df["sex"] = df.defendant_sex.map({"Male": "Male", "Female": "Female"}).fillna("Unknown")
df["pend"] = df.outcome.eq("pending")

w = df[df.filing_date.dt.year.between(*YEARS)]
print("source in window:", w.source_prr.value_counts().to_dict())
print("severity unknown in window:", pct(w.sev_class.eq("Unknown").mean()), "%")
print("defendant id missing in window:", pct(w.defendant_id.isna().mean()), "%")
print(df.race.value_counts().to_string())

In [ ]:
OUTCOME_RANK = {"trial_conviction": 8, "plea": 7, "cwof": 6, "acquittal": 5, "diversion": 4,
                "dismissal": 3, "declined": 2, "other": 1, "pending": 0}

lead = (df.sort_values(["case_key", "sev_rank"], ascending=[True, False])
          .drop_duplicates("case_key").set_index("case_key")
          [[DEF_COL, "sev_rank", "sev_class", "offense_cat", "race", "sex",
            "charge_description", "court_name", "data_quality_tier"]])
agg = df.groupby("case_key").agg(filed=("filing_date", "min"), dispo=("disposition_date", "max"),
                                 n_charges=("case_key", "size"),
                                 all_civil=("is_civil", "all"), any_pending=("pend", "any"))
cr = df[~df.is_civil]
outcome = (cr.assign(r=cr.outcome.map(OUTCOME_RANK)).sort_values("r", ascending=False)
             .drop_duplicates("case_key").set_index("case_key").outcome)

cases_all = lead.join(agg).join(outcome)
cases_all["outcome"] = cases_all.outcome.fillna("civil")
cases_all["any_lesser_plea"] = cases_all.index.isin(cr.loc[cr.outcome.eq("plea") & cr.lesser, "case_key"].unique())
cases_all["year"] = cases_all.filed.dt.year
dd = (cases_all.dispo - cases_all.filed).dt.days
cases_all["days_to_dispo"] = dd.where((dd >= 0) & ~cases_all.any_pending)

# prior record: excludes migration artifacts and null ids (insufficient for ~half the window)
p = cases_all[~cases_all.all_civil & cases_all.data_quality_tier.ne("migration_artifact")
              & cases_all[DEF_COL].notna()].sort_values("filed")
cases_all["prior_cases"] = p.groupby(DEF_COL).cumcount().reindex(cases_all.index)
cases_all["prior_level"] = (pd.cut(cases_all.prior_cases, [-1, 0, 2, np.inf], labels=["0", "1-2", "3+"])
                              .astype(object).fillna("Unknown"))

cases = cases_all[cases_all.year.between(*YEARS)]
ch_w = df[df.filing_date.dt.year.between(*YEARS)]
crim = cases[~cases.all_civil & cases.outcome.ne("pending")]
charged = crim[crim.outcome.ne("declined")]
convicted = charged[charged.outcome.isin(["plea", "trial_conviction"])]
print(len(cases), "cases |", len(crim), "criminal disposed |", len(charged), "charged |", len(convicted), "convicted")

In [ ]:
SENT_MAP = {
    "Prison Sentence - Committed": "Incarceration", "Prison Sentence - Concurrent": "Incarceration",
    "Prison Sentence - On and After": "Incarceration", "Prison Sentence - Consecutive": "Incarceration",
    "Prison Sentence - Life without Parole": "Incarceration", "Prison Sentence - Life with Parole": "Incarceration",
    "Prison/Probation - Split Sentence": "Incarceration",
    "Prison - DYS until 18th Birthday": "Incarceration", "Prison - DYS until 19th Birthday": "Incarceration",
    "Prison - DYS until 21st Birthday": "Incarceration", "Prison - DYS until Age 21, then HOC suspended": "Incarceration",
    "Suspended Prison Sentence": "Probation (suspended)", "Suspended until 18th Birthday": "Probation (suspended)",
    "Suspended until 21st Birthday": "Probation (suspended)", "Suspended Sentence On & After": "Probation (suspended)",
    "Probation  (see Probation tab)": "Probation", "Probation (Recommended)": "Probation",
    "Probation until 18th Birthday": "Probation", "Probation until 19th Birthday": "Probation",
    "Fined (Fines & Surfines)": "Fine/other", "Court Costs  (see Costs/Victim tab)": "Fine/other",
    "Restitution  (see Costs/Victim tab)": "Fine/other", "Community Service": "Fine/other",
    "Special Program": "Fine/other",
    "Filed": "Filed (no sentence)",
    "Cont'd w/o Finding (see Probation tab)": "CWOF", "C.W.O.F. (Recommended)": "CWOF",
}
conv_all = df[df.outcome.isin(["plea", "trial_conviction"]) & ~df.is_civil].copy()
conv_all["sent"] = conv_all.sentence_type_description.map(SENT_MAP)
conv_all.loc[conv_all.sentence_type_description.isna(), "sent"] = "Missing"
print("unmapped:", conv_all.loc[conv_all.sent.isna(), "sentence_type_description"].unique())
conv_all["sent"] = conv_all.sent.fillna("Other")

cv_w = conv_all[conv_all.filing_date.dt.year.between(*YEARS)]
sent_missing = cv_w.sent.eq("Missing").mean()
HAS_INCARC = sent_missing <= MISSING_MAX
print(f"sentence missing among convictions: {100 * sent_missing:.1f}% -> HAS_INCARC={HAS_INCARC}")
print(cv_w.sent.value_counts().to_string())

rank = {"Incarceration": 6, "Probation (suspended)": 5, "Probation": 5, "Fine/other": 4,
        "CWOF": 3, "Filed (no sentence)": 2, "Other": 1, "Missing": 0}
case_sent = (conv_all.assign(r=conv_all.sent.map(rank)).sort_values("r", ascending=False)
                     .drop_duplicates("case_key").set_index("case_key").sent)

In [ ]:
yr = ch_w.filing_date.dt.year
groups = [
    ({"defendant_id": ch_w.defendant_id.isna(), "filing_date": ch_w.filing_date.isna(),
      "disposition_date": ch_w.disposition_date.isna(),
      "charge_code": ch_w.charge_code.isna() | ch_w.placeholder,
      "disposition_code": ch_w.disposition_code.isna(),
      "race": ch_w.race.eq("Unknown"), "sex": ch_w.sex.eq("Unknown"),
      "court": ch_w.court_name.isna(), "severity": ch_w.sev_class.eq("Unknown"),
      "offense_group": ch_w.offense_cat.eq("Other/Unknown")}, yr),
    ({"sentence_type (convictions)": cv_w.sent.eq("Missing")}, cv_w.filing_date.dt.year),
]
rows = []
for checks, yrs in groups:
    for k, miss in checks.items():
        by_year = (~miss).groupby(yrs).mean()
        rows.append(dict(field=k, pct_missing=pct(miss.mean()),
                         years_available=",".join(str(int(y)) for y, v in by_year.items() if v > 0),
                         status="ok" if miss.mean() <= MISSING_MAX else "insufficient data"))
rows += [dict(field=k, pct_missing=100.0, years_available="", status=f"insufficient data ({why})")
         for k, why in [("sentence_length", "impsd_* columns 100% null"),
                        ("dob/age", "not delivered"), ("bail/detention", "not delivered")]]
coverage = pd.DataFrame(rows)
coverage

In [ ]:
w_cases = cases[~cases.all_civil]
print(pd.crosstab(w_cases.court_name, w_cases.year))
print(cases_all[~cases_all.all_civil].groupby("year").size().loc[2012:2024])   # WADADAM vs PRR transition

In [ ]:
DISTRICT_ONLY = True
if DISTRICT_ONLY:
    cases = cases[cases.court_name.str.contains("District", na=False)]
    ch_w = ch_w[ch_w.court_name.str.contains("District", na=False)]